# ERIS drift detector — GPU Colab run

This notebook is meant to be executed later on a GPU Colab runtime, not on the current local machine. It sets up the repository, loads `SAEProbe`, and runs a small real-model drift experiment.

Recommended runtime:
- Colab GPU runtime
- high-RAM if available
- use a Gemma-compatible Hugging Face token if the model access requires it


In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get('LATENT_RELAY_REPO_URL', 'https://github.com/ArthurVigier/latent-relay.git')
REPO_DIR = Path('/content/latent-relay')

if not REPO_DIR.exists():
    !git clone $REPO_URL /content/latent-relay

%cd /content/latent-relay
!python -m pip install -q --upgrade pip
!python -m pip install -q numpy scipy matplotlib pandas pytest transformers accelerate sentencepiece huggingface_hub sae-lens "transformer-lens>=3.0.0b0"


In [ ]:
import torch

assert torch.cuda.is_available(), 'Switch Colab runtime to GPU before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA available:', torch.cuda.is_available())


In [ ]:
# Optional: set your token in Colab secrets or env before running.
# os.environ["HF_TOKEN"] = "..."
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face")
else:
    print("HF_TOKEN not set; continue only if public access works for your chosen model.")


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from eris.drift_detector import DriftDetector
from eris.sae_probe import SAEProbe


In [ ]:
MODEL_ID = "google/gemma-3-9b-it"
LAYERS = [10, 20, 30]
TEXTS = [
    "Find the value of x if 2x + 7 = 19.",
    "Find the value of x if 2x + 7 = 19. First isolate the variable and verify the result.",
    "Find the value of x if 2x + 7 = 19. Now explain why the solution is unique and give a short proof.",
]

probe = SAEProbe(
    model_id=MODEL_ID,
    layers=LAYERS,
    sae_width="16k",
    l0="medium",
    device="cuda",
)

detector = DriftDetector(
    threshold=0.30,
    window=2,
    jaccard_weight=0.7,
    cosine_weight=0.3,
    layer_weights={10: 1.0, 20: 2.0, 30: 1.5},
    comparison_mode="previous",
)


In [ ]:
reference = probe.probe(TEXTS[0], top_k=20)
detector.register_reference(reference)

reports = []
for step, text in enumerate(TEXTS[1:], start=1):
    current = probe.probe(text, top_k=20)
    report = detector.compute_drift(current, step=step)
    reports.append(report)
    print(report.summary)
    print("-" * 80)


In [ ]:
payload = [report.to_dict() for report in reports]
steps = [item["step"] for item in payload]
raw_scores = [item["raw_drift_score"] for item in payload]
smooth_scores = [item["drift_score"] for item in payload]

plt.figure(figsize=(8, 4))
plt.plot(steps, raw_scores, marker='o', label='raw drift')
plt.plot(steps, smooth_scores, marker='s', label='smoothed drift')
plt.axhline(detector.threshold, color='red', linestyle='--', label='threshold')
plt.title('ERIS drift on real SAEProbe outputs')
plt.xlabel('step')
plt.ylabel('score')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

out_path = Path('/content/drift_gpu_reports.json')
out_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print('Saved:', out_path)


In [ ]:
!pytest -q test_drift_detector.py test_orchestrator_drift_formatting.py
